# **ReneWind - Predictive Maintenance for Wind Turbine Generators**
## *Introduction to Neural Networks - Full Code Project*

# **Problem Statement**

## Business Context

Renewable energy sources play an increasingly important role in the global energy mix, as the effort to reduce the environmental impact of energy production increases.

Out of all the renewable energy alternatives, wind energy is one of the most developed technologies worldwide. The U.S Department of Energy has put together a guide to achieving operational efficiency using predictive maintenance practices.

Predictive maintenance uses sensor information and analysis methods to measure and predict degradation and future component capability. The idea behind predictive maintenance is that failure patterns are predictable and if component failure can be predicted accurately and the component is replaced before it fails, the costs of operation and maintenance will be much lower.

The sensors fitted across different machines involved in the process of energy generation collect data related to various environmental factors (temperature, humidity, wind speed, etc.) and additional features related to various parts of the wind turbine (gearbox, tower, blades, break, etc.).

## Objective

"ReneWind" is a company working on improving the machinery/processes involved in the production of wind energy using machine learning and has collected data of generator failure of wind turbines using sensors. They have shared a ciphered version of the data, as the data collected through sensors is confidential. The data has **40 predictors, 20000 observations in the training set and 5000 in the test set**.

The objective is to build various classification models, tune them, and find the best one that will help identify failures so that the generators could be repaired before failing/breaking to reduce the overall maintenance cost.

The nature of predictions made by the classification model will translate as follows:

- **True Positives (TP)** are failures correctly predicted by the model. These will result in **repair costs**.
- **False Negatives (FN)** are real failures where there is no detection by the model. These will result in **replacement costs**.
- **False Positives (FP)** are detections where there is no failure. These will result in **inspection costs**.

It is given that the **cost of repairing a generator is much less than the cost of replacing it**, and the **cost of inspection is less than the cost of repair**.

`1` in the target variable should be considered as `failure` and `0` represents `No failure`.

## Data Description

The data provided is a transformed version of the original data which was collected using sensors.

- `Train.csv` - To be used for training and tuning of models.
- `Test.csv` - To be used **only** for testing the performance of the final best model.

Both the datasets consist of 40 predictor variables (`V1` to `V40`) and 1 target variable (`Target`).

# **Installing and Importing the necessary libraries**

In [1]:
# Installing the libraries with the specified version
!pip install --no-deps tensorflow==2.19.0 scikit-learn==1.6.1 matplotlib===3.10.0 seaborn==0.13.2 numpy==2.0.2 pandas==2.2.2 -q --user --no-warn-script-location

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [2]:
# ----------------------------------------------------------------------
# Data manipulation
# ----------------------------------------------------------------------
import numpy as np
import pandas as pd

# ----------------------------------------------------------------------
# Visualisation
# ----------------------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns

# ----------------------------------------------------------------------
# Preprocessing and model evaluation (scikit-learn)
# ----------------------------------------------------------------------
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    recall_score,
    precision_score,
    f1_score,
    accuracy_score,
)

# ----------------------------------------------------------------------
# Deep learning (TensorFlow / Keras)
# ----------------------------------------------------------------------
import tensorflow as tf
from tensorflow.keras import backend
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import SGD, Adam

# ----------------------------------------------------------------------
# Housekeeping: suppress warnings and fix the plotting style
# ----------------------------------------------------------------------
import warnings, os, random
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"          # hide TensorFlow info/warning logs
tf.get_logger().setLevel("ERROR")

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 100

print("NumPy      :", np.__version__)
print("Pandas     :", pd.__version__)
print("TensorFlow :", tf.__version__)

### Setting the random seed for reproducibility

Neural networks are initialised with random weights, and mini-batches are shuffled randomly. Without fixing the seeds, re-running this notebook would produce slightly different numbers each time. We define a helper below and call it before building every model so that all our comparisons are apples-to-apples.

In [3]:
def reset_seed(seed=42):
    """Clear the Keras session and fix all random seeds so that every model
    below starts from an identical, reproducible initial state."""
    backend.clear_session()
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


reset_seed()
print("Seeds fixed - results in this notebook are reproducible.")

# **Loading the Data**

In [4]:
# Uncomment and run the following two lines if you are on Google Colab
# from google.colab import drive
# drive.mount('/content/drive')

# Update the paths below to point to the location of your files
train_path = "Train.csv"
test_path = "Test.csv"

# Loading the training and the test data
train_raw = pd.read_csv(train_path)
test_raw = pd.read_csv(test_path)

# Working on copies so that the original data frames stay untouched
data = train_raw.copy()
test = test_raw.copy()

print("Training data shape :", data.shape)
print("Test data shape     :", test.shape)

# **Data Overview**

Before any modelling, we inspect the data to answer five questions:

1. What do the rows look like?
2. What are the data types and how much memory does the data use?
3. Are there missing values?
4. Are there duplicate rows?
5. How is the target distributed - i.e. how imbalanced is the problem?

### Viewing the first and last few rows of the training data

In [5]:
# First five rows
data.head()

In [6]:
# Last five rows
data.tail()

**Observations**

- The data contains 40 predictor columns named `V1` to `V40` and a single binary column `Target`.
- All predictors are continuous numeric values, already centred close to zero and both positive and negative - consistent with the statement that the data is a *ciphered / transformed* version of the raw sensor readings.
- Because the variables are anonymised, we cannot attach physical meaning (temperature, wind speed, gearbox vibration, etc.) to any individual column. Our analysis therefore has to be purely statistical, and our business recommendations will be framed around *model behaviour and cost*, not around individual sensors.

### Checking the shape and structure of the data

In [7]:
# Shape of the training and test data
print(f"The training data has {data.shape[0]} rows and {data.shape[1]} columns.")
print(f"The test data has     {test.shape[0]} rows and {test.shape[1]} columns.")

In [8]:
# Structure, data types and memory usage
data.info()

**Observations**

- The training set has **20,000 rows and 41 columns**, and the test set has **5,000 rows and 41 columns** - exactly as described in the problem statement.
- All 40 predictors are of type `float64` and the `Target` is `int64`. **There are no categorical or text columns**, so no encoding of predictors is required.
- `V1` and `V2` show fewer than 20,000 non-null entries, which tells us these two columns carry missing values. Every other column is complete.

### Statistical summary of the data

In [9]:
# Statistical summary of the predictors (transposed for readability)
data.describe().T

**Observations**

- The means of all predictors sit close to zero (roughly between -3.6 and +2.5), confirming the data has already been centred by the transformation applied by ReneWind.
- **The standard deviations, however, differ substantially - from 1.65 (`V22`) to 5.50 (`V32`), a spread of more than three times.** The ranges are similarly uneven, with some variables spanning roughly -8 to +8 and others -20 to +24.
- This spread in scale is important. A neural network trained with gradient descent converges far more slowly, and can be dominated by the highest-variance inputs, when features are on different scales. **Standardisation is therefore a mandatory preprocessing step**, which we perform later (after the train/validation split, to avoid data leakage).
- Minimum and maximum values are far from the quartiles for several variables, suggesting the presence of extreme values. Since these are sensor readings from machinery that is *expected to sometimes behave abnormally*, these extreme readings are likely to be genuine signal about impending failure rather than data-entry errors. **We will therefore not remove or cap them.**

### Checking for missing values

In [10]:
# Count and percentage of missing values per column - training data
missing_train = pd.DataFrame({
    "Missing_Count": data.isnull().sum(),
    "Missing_Percent": (data.isnull().sum() / len(data) * 100).round(4),
})
print("Columns with missing values in the TRAINING data:")
display(missing_train[missing_train["Missing_Count"] > 0])

# Count and percentage of missing values per column - test data
missing_test = pd.DataFrame({
    "Missing_Count": test.isnull().sum(),
    "Missing_Percent": (test.isnull().sum() / len(test) * 100).round(4),
})
print("\nColumns with missing values in the TEST data:")
display(missing_test[missing_test["Missing_Count"] > 0])

**Observations**

- Only two columns contain missing values: **`V1` (18 missing) and `V2` (18 missing) in the training data**, and **`V1` (5) and `V2` (6) in the test data**.
- This amounts to **less than 0.1% of the rows** in each case - a negligible fraction.
- Since the volume is so small, dropping the rows would also be defensible, but **imputation is safer**: it keeps all 20,000 training observations (valuable given only 1,110 of them are failures) and, critically, we *cannot* drop rows from the test set because we are required to score every test observation.
- We will impute with the **median**, which is robust to the extreme values noted above. Crucially, the imputer will be **fitted on the training split only** and then applied to the validation and test sets - see the Data Preprocessing section.

### Checking for duplicate rows

In [11]:
print("Number of duplicate rows in the training data :", data.duplicated().sum())
print("Number of duplicate rows in the test data     :", test.duplicated().sum())

**Observations**

- There are **no duplicate rows** in either dataset, so no de-duplication is required. Each row represents a distinct sensor observation.

# **Exploratory Data Analysis**

## Univariate analysis

### Distribution of the target variable

This is the single most important chart in the EDA, because the balance of the target dictates our choice of evaluation metric and our modelling strategy.

In [12]:
# Absolute counts and proportions of the target
target_counts = data["Target"].value_counts().sort_index()
target_perc = data["Target"].value_counts(normalize=True).sort_index() * 100

summary = pd.DataFrame({
    "Count": target_counts,
    "Percentage": target_perc.round(2),
})
summary.index = ["0 - No Failure", "1 - Failure"]
display(summary)

# Visualising the imbalance
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

sns.countplot(x="Target", data=data, ax=ax[0], hue="Target", legend=False)
for container in ax[0].containers:
    ax[0].bar_label(container, fmt="%d", padding=3)
ax[0].set_title("Class distribution of the Target (Training data)")
ax[0].set_xlabel("Target (0 = No Failure, 1 = Failure)")
ax[0].set_ylabel("Number of observations")

ax[1].pie(
    target_counts, labels=["No Failure (0)", "Failure (1)"],
    autopct="%1.2f%%", startangle=90, explode=(0, 0.12),
    colors=["#4C72B0", "#C44E52"],
)
ax[1].set_title("Proportion of failures vs non-failures")

plt.tight_layout()
plt.show()

**Observations**

- The target is **severely imbalanced**: only **1,110 out of 20,000 observations (5.55%) are generator failures**, while 18,890 (94.45%) are healthy readings.
- This has three direct consequences for the rest of the project:
  1. **Accuracy is a misleading metric.** A trivial model that predicts "no failure" for every turbine would score 94.45% accuracy while catching zero failures - and would be worthless to ReneWind.
  2. **The train/validation split must be stratified** (`stratify=y`), otherwise random chance could leave the validation set with a materially different failure rate and make our comparisons unreliable.
  3. **The minority class needs help during training.** With ~19 healthy readings for every failure, a network minimising plain cross-entropy will be strongly biased towards predicting the majority class. We will address this using **class weights** in several of our models.
- Reassuringly, the imbalance in the **test set is 5.64%**, essentially identical to the training set, so the two datasets are drawn from the same distribution.

### Distribution of the predictor variables

With 40 anonymised predictors we cannot inspect them one at a time in narrative form. Instead we plot all 40 distributions on a single grid to check their overall shape, and then use boxplots to examine spread and extreme values.

In [13]:
# List of the 40 predictor columns
predictors = [col for col in data.columns if col != "Target"]

# Histograms with KDE for all 40 predictors
fig, axes = plt.subplots(8, 5, figsize=(20, 26))
axes = axes.flatten()

for i, col in enumerate(predictors):
    sns.histplot(data=data, x=col, kde=True, ax=axes[i], color="#4C72B0", bins=40)
    axes[i].axvline(data[col].mean(), color="red", linestyle="--", linewidth=1.2)
    axes[i].set_title(f"{col}  (skew = {data[col].skew():.2f})", fontsize=10)
    axes[i].set_xlabel("")
    axes[i].set_ylabel("")

fig.suptitle("Distribution of all 40 predictor variables (red dashed line = mean)",
             fontsize=16, y=1.001)
plt.tight_layout()
plt.show()

**Observations**

- **Nearly every predictor is approximately bell-shaped (roughly normal) and unimodal.** This is expected: the variables are a transformed / ciphered version of the original sensor readings, and the transformation has evidently centred and symmetrised them.
- The skewness values printed in each title are almost all close to zero (broadly within -0.5 to +0.5), confirming near-symmetry. **No predictor requires a log or power transformation.**
- The distributions are centred at different points and have visibly different widths - reinforcing the need for standardisation before feeding them into a neural network.
- There is no evidence of the multi-modal or spiked distributions that would suggest a hidden categorical variable encoded as a number.

In [14]:
# Boxplots for all 40 predictors to visualise spread and extreme values
fig, axes = plt.subplots(8, 5, figsize=(20, 24))
axes = axes.flatten()

for i, col in enumerate(predictors):
    sns.boxplot(data=data, x=col, ax=axes[i], color="#55A868", fliersize=1.5)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel("")

fig.suptitle("Boxplots of all 40 predictor variables", fontsize=16, y=1.001)
plt.tight_layout()
plt.show()

In [15]:
# Quantifying the proportion of extreme values using the IQR rule
outlier_summary = []
for col in predictors:
    q1, q3 = data[col].quantile(0.25), data[col].quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    pct = ((data[col] < low) | (data[col] > high)).mean() * 100
    outlier_summary.append({"Variable": col, "Outlier_Percent": round(pct, 2)})

outlier_df = pd.DataFrame(outlier_summary).sort_values("Outlier_Percent", ascending=False)
print("Top 10 variables by proportion of IQR-rule outliers:")
display(outlier_df.head(10).reset_index(drop=True))
print(f"\nAverage proportion of outliers across all 40 predictors: "
      f"{outlier_df['Outlier_Percent'].mean():.2f}%")

**Observations**

- Every predictor shows some values beyond the 1.5 x IQR whiskers, but the proportion is modest - **averaging 1.25% per variable and peaking at 4.01% (`V34`), followed by `V18` at 3.66%**. For reference, a perfectly normal variable produces about 0.70% of such points, so these counts are only mildly above what normality alone would generate.
- **These extreme readings should not be treated as errors and must not be removed.** In a predictive-maintenance context, an unusually high vibration or temperature reading is precisely the *signal* that a generator is degrading. Removing them would strip out the very information the model needs to detect the 5.55% of failure cases.
- Their presence does, however, justify using the **median (not the mean) for imputing `V1` and `V2`**, and standardisation rather than min-max scaling.

## Bivariate Analysis

### Correlation between the predictors

Highly correlated inputs carry redundant information. While neural networks tolerate multicollinearity far better than linear models do, strong redundancy still inflates the number of parameters the network has to learn.

In [16]:
# Correlation matrix of the predictors
corr = data[predictors].corr()

plt.figure(figsize=(19, 15))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="coolwarm", center=0, vmin=-1, vmax=1,
            annot=True, fmt=".2f", annot_kws={"size": 6},
            linewidths=0.4, cbar_kws={"shrink": 0.6})
plt.title("Correlation heatmap of the 40 predictor variables", fontsize=15)
plt.tight_layout()
plt.show()

In [17]:
# Listing the strongest pairwise correlations
corr_pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .reset_index()
)
corr_pairs.columns = ["Variable_1", "Variable_2", "Correlation"]
corr_pairs["Abs_Correlation"] = corr_pairs["Correlation"].abs()
top_corr = corr_pairs.sort_values("Abs_Correlation", ascending=False).head(15)

print("Top 15 strongest pairwise correlations among the predictors:")
display(top_corr.drop(columns="Abs_Correlation").reset_index(drop=True))

print(f"\nNumber of predictor pairs with |correlation| > 0.70 : "
      f"{(corr_pairs['Abs_Correlation'] > 0.70).sum()}")
print(f"Number of predictor pairs with |correlation| > 0.50 : "
      f"{(corr_pairs['Abs_Correlation'] > 0.50).sum()}")

**Observations**

- Out of the 780 possible predictor pairs, **22 pairs exceed a correlation of 0.70 and 119 pairs exceed 0.50** - so roughly 15% of pairs carry a moderate or stronger linear relationship, while the large majority remain weakly related.
- The strongest relationships are substantial: **`V7`-`V15` at 0.87, `V2`-`V14` at -0.85, `V16`-`V21` at 0.84, and `V24`-`V32` at 0.83**. Both positive and negative strong pairs appear.
- These clusters most likely correspond to sensors mounted on the same subsystem of the turbine - for example two readings from the same gearbox or blade assembly - which would naturally rise and fall together. A negative pair such as `V2`-`V14` would then represent two ends of the same physical trade-off.
- **This level of redundancy would be a serious problem for logistic regression, where multicollinearity destabilises the coefficients. It is not a problem for a neural network**, which learns its own internal representation and is not required to attribute a unique, interpretable weight to each input. We therefore **retain all 40 predictors** rather than dropping any.
- **We will not perform dimensionality reduction (e.g. PCA)**, for two reasons. First, PCA is an **unsupervised** transform: it selects components that maximise total variance, which is not the same thing as class separation, so it can discard low-variance directions that happen to carry the failure signal. Second, and more decisively, **the first hidden layer of our network already applies a learned linear projection of the 40 inputs** - a supervised, task-optimised version of exactly what PCA would do blindly. Running PCA first would replace a projection optimised for detecting failures with one optimised for preserving variance.

### Relationship between the predictors and the Target

This is the part of the EDA that tells us whether the problem is learnable at all. If the failure and non-failure groups had identical distributions on every predictor, no model - however deep - could separate them.

In [18]:
# Standardised difference in means between the failure and non-failure groups.
# We use Cohen's d, which expresses the gap in means in units of pooled standard
# deviation, so that variables on different scales can be compared fairly.
fail = data[data["Target"] == 1]
nofail = data[data["Target"] == 0]

effects = []
for col in predictors:
    m1, m0 = fail[col].mean(), nofail[col].mean()
    s1, s0 = fail[col].std(), nofail[col].std()
    n1, n0 = len(fail), len(nofail)
    pooled = np.sqrt(((n1 - 1) * s1**2 + (n0 - 1) * s0**2) / (n1 + n0 - 2))
    effects.append({
        "Variable": col,
        "Mean_NoFailure": round(m0, 3),
        "Mean_Failure": round(m1, 3),
        "Cohens_d": round((m1 - m0) / pooled, 3),
    })

effect_df = pd.DataFrame(effects)
effect_df["Abs_d"] = effect_df["Cohens_d"].abs()
effect_df = effect_df.sort_values("Abs_d", ascending=False).reset_index(drop=True)

print("Top 12 predictors that best separate failures from non-failures:")
display(effect_df.head(12).drop(columns="Abs_d"))

# Visualising the effect sizes for all 40 predictors
plt.figure(figsize=(13, 8))
plot_df = effect_df.sort_values("Cohens_d")
colors = ["#C44E52" if v > 0 else "#4C72B0" for v in plot_df["Cohens_d"]]
plt.barh(plot_df["Variable"], plot_df["Cohens_d"], color=colors)
plt.axvline(0, color="black", linewidth=0.9)
plt.axvline(0.2, color="grey", linestyle="--", linewidth=0.9)
plt.axvline(-0.2, color="grey", linestyle="--", linewidth=0.9)
plt.xlabel("Cohen's d  (standardised difference in means: Failure minus No-Failure)")
plt.title("How strongly does each predictor separate failures from non-failures?", fontsize=14)
plt.tight_layout()
plt.show()

**Observations**

- **Many predictors show a strong, systematic shift between failing and healthy generators.** The leading separators reach Cohen's *d* values above 1.0 - `V18` (-1.34), `V21` (+1.16), `V15` (+1.12), `V7` (+1.07), `V16` (+1.04), `V39` (-1.02) - which by convention is a **large** effect, far beyond the 0.2 "small effect" threshold marked by the dashed grey lines.
- Both directions are represented: some variables read **higher** on failing generators (red bars) and others read **lower** (blue bars). This is consistent with a physical system where a developing fault raises some readings (e.g. vibration, bearing temperature) while suppressing others (e.g. output efficiency, rotational smoothness).
- The failure signal is therefore **genuinely present and reasonably strong** - this problem is learnable, and we should expect a well-tuned network to do materially better than chance.
- Critically, though, **no single predictor separates the classes cleanly on its own.** Even at *d* = 1.34 the two distributions still overlap across most of their range, so any single-sensor threshold rule would misclassify a large share of turbines. The failure signature lives in the *combination* of many sensors interacting non-linearly - which is exactly the situation a multi-layer neural network is designed for, and justifies our modelling approach over a simple rule-based alarm.

In [19]:
# Distribution of the eight most discriminative predictors, split by Target
top8 = effect_df["Variable"].head(8).tolist()

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for i, col in enumerate(top8):
    sns.kdeplot(data=data, x=col, hue="Target", fill=True, alpha=0.4,
                common_norm=False, ax=axes[i], palette={0: "#4C72B0", 1: "#C44E52"})
    axes[i].set_title(f"{col}   (Cohen's d = {effect_df.loc[effect_df['Variable']==col, 'Cohens_d'].values[0]})",
                      fontsize=11)
    axes[i].set_ylabel("Density")

fig.suptitle("Distributions of the 8 most discriminative predictors, by failure status",
             fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

In [20]:
# Boxplots of the same eight predictors against the Target
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for i, col in enumerate(top8):
    sns.boxplot(data=data, x="Target", y=col, ax=axes[i], hue="Target",
                legend=False, palette={0: "#4C72B0", 1: "#C44E52"}, fliersize=1.5)
    axes[i].set_title(col, fontsize=11)
    axes[i].set_xlabel("Target (0 = No Failure, 1 = Failure)")

fig.suptitle("Boxplots of the 8 most discriminative predictors, by failure status",
             fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

**Observations**

- The density plots make the overlap explicit: for every one of the eight strongest predictors, the red (failure) curve is **shifted** relative to the blue (no-failure) curve, but the two curves still overlap across most of their range.
- The boxplots tell the same story through the medians and quartiles - the failure group sits consistently above or below the healthy group, and the failure group generally shows a **wider spread**, which is intuitive: healthy machines behave consistently, degrading machines behave erratically.
- **Business takeaway:** a maintenance engineer could not reliably diagnose a failing generator by watching any single sensor dial, because a large share of failing turbines produce readings that look perfectly normal on that dial. A model that fuses all 40 signals simultaneously is required, which is precisely what we build next.

### Summary of EDA findings

| # | Finding | Consequence for modelling |
|---|---|---|
| 1 | 20,000 training rows, 40 continuous anonymised predictors, no categorical columns | No encoding needed; all 40 features go straight into the network |
| 2 | Target is severely imbalanced - only **5.55%** failures | Accuracy is unusable; **Recall** becomes the metric of choice; use `stratify` and class weights |
| 3 | Only `V1` and `V2` have missing values, **< 0.1%** of rows | Median imputation, fitted on the training split only |
| 4 | No duplicate rows | No de-duplication needed |
| 5 | Predictors are near-normal but on **very different scales** (std 1.65 to 5.50) | **Standardisation is mandatory** before training |
| 6 | Extreme values present in every column, averaging **1.25%** per variable | **Retained** - they are genuine degradation signal, not noise |
| 7 | 22 of 780 predictor pairs correlate above 0.70 (max **0.87**, `V7`-`V15`); the majority are weak | Keep all 40 features - neural networks are robust to multicollinearity; no PCA, no feature dropping |
| 8 | Several predictors separate the classes with **large effect sizes** (max Cohen's *d* = **1.34**, `V18`), yet none separates them cleanly alone | The signal is real and learnable, but requires a **multi-layer non-linear model**, not threshold rules |

# **Data Preprocessing**

Our EDA identified exactly three preprocessing requirements, and nothing more:

| Requirement | Reason from EDA | Treatment |
|---|---|---|
| Missing values in `V1`, `V2` | 18 rows each (< 0.1%) | **Median** imputation (robust to the extreme values we found) |
| Predictors on very different scales | Std deviations range 1.65 to 5.50 | **Standardisation** (`StandardScaler`) |
| Severe class imbalance (5.55% failures) | Would bias the network to the majority class | **Stratified** split now; **class weights** during training later |

Notably, there is **no encoding step**: all 40 predictors are already continuous numeric, and the target is already binary 0/1.

## Avoiding data leakage - why the order of operations matters

**Data leakage** occurs when information from data the model is supposed to be evaluated on bleeds into the data it is trained on. The result is a validation score that flatters the model and collapses in production.

Both of our preprocessing steps *learn a parameter from the data*, which makes them leakage risks:

- `SimpleImputer(strategy="median")` learns a **median** for `V1` and `V2`.
- `StandardScaler` learns a **mean and standard deviation** for each of the 40 columns.

If we computed those statistics across all 20,000 rows and *then* split, the validation rows would have contributed to the numbers used to transform the training rows. The validation set would no longer be an honest stand-in for unseen data.

**We therefore follow this strict order, and never deviate from it:**

1. **Split first** - carve `Train.csv` into a training set and a validation set, stratified on the target.
2. **`fit` on the training split only** - the imputer and the scaler see the training rows and nothing else.
3. **`transform` everything** - apply those frozen, training-derived parameters to the training split, the validation split, and the held-out `Test.csv`.

Note that `Test.csv` is treated exactly like the validation set: it is only ever passed through `.transform()`, never `.fit()`. It is opened once, at the very end, to score the single final model.

### Step 2.1 - Separating the predictors from the target

In [21]:
# Separating the predictors (X) from the target (y) in the training data
X = data.drop("Target", axis=1)
y = data["Target"]

# Doing the same for the held-out test data
X_test = test.drop("Target", axis=1)
y_test = test["Target"]

print("Predictors (X)  :", X.shape)
print("Target (y)      :", y.shape)
print("Test predictors :", X_test.shape)
print("Test target     :", y_test.shape)

### Step 2.2 - Splitting into training and validation sets (BEFORE any transformation)

We hold back 20% of `Train.csv` as a validation set for model comparison, and pass `stratify=y` so that both splits preserve the 5.55% failure rate. Without stratification, random chance could hand us a validation set with a materially different failure rate, making every model comparison in this notebook unreliable.

In [22]:
# Stratified 80/20 split - performed FIRST, before imputation or scaling
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.20,
    random_state=1,
    stratify=y,      # preserves the failure rate in both splits
)

print(f"Training set   : {X_train.shape[0]} rows")
print(f"Validation set : {X_val.shape[0]} rows")
print(f"Test set       : {X_test.shape[0]} rows")

# Verifying that stratification preserved the class balance
balance = pd.DataFrame({
    "Training": y_train.value_counts(normalize=True).sort_index() * 100,
    "Validation": y_val.value_counts(normalize=True).sort_index() * 100,
    "Test": y_test.value_counts(normalize=True).sort_index() * 100,
}).round(2)
balance.index = ["0 - No Failure (%)", "1 - Failure (%)"]

print("\nClass balance across the three sets:")
display(balance)

print(f"\nFailure counts -> Training: {y_train.sum()}   "
      f"Validation: {y_val.sum()}   Test: {y_test.sum()}")

**Observations**

- The 20,000 training observations are split into **16,000 for training and 4,000 for validation**, with `Test.csv` supplying a further 5,000 completely untouched observations.
- Stratification worked as intended: the failure rate is **5.55% in the training split and 5.55% in the validation split**, matching the 5.55% of the original data and closely matching the 5.64% in the test set.
- In absolute terms the training split contains **888 failures** and the validation split **222 failures**. 222 positive cases is a workable but not generous number - it means each individual failure in the validation set moves recall by roughly 0.45 percentage points, so we should not read too much into differences between models of less than about 1 point.

### Step 2.3 - Missing value treatment (imputer fitted on the training split only)

In [23]:
# Confirming which columns need imputation and their pre-treatment state
print("Missing values BEFORE imputation")
print("  Training split   :", X_train.isnull().sum().sum(), "total")
print("  Validation split :", X_val.isnull().sum().sum(), "total")
print("  Test set         :", X_test.isnull().sum().sum(), "total")

# Fitting the median imputer on the TRAINING SPLIT ONLY - this is the key
# leakage-prevention step. The medians below are learned from 16,000 rows.
imputer = SimpleImputer(strategy="median")
imputer.fit(X_train)

print("\nMedians learned from the training split (for the two affected columns):")
learned = pd.Series(imputer.statistics_, index=X_train.columns)
display(learned[["V1", "V2"]].to_frame("Training median").round(4))

# Applying those frozen medians to all three sets
X_train = pd.DataFrame(imputer.transform(X_train), columns=X_train.columns, index=X_train.index)
X_val   = pd.DataFrame(imputer.transform(X_val),   columns=X_val.columns,   index=X_val.index)
X_test  = pd.DataFrame(imputer.transform(X_test),  columns=X_test.columns,  index=X_test.index)

print("\nMissing values AFTER imputation")
print("  Training split   :", X_train.isnull().sum().sum(), "total")
print("  Validation split :", X_val.isnull().sum().sum(), "total")
print("  Test set         :", X_test.isnull().sum().sum(), "total")

**Observations**

- All missing values have been filled. The medians used were learned **exclusively from the 16,000 training rows** and then applied unchanged to the validation and test sets - so no information from the evaluation data influenced the training data.
- Because fewer than 0.1% of values were missing, this treatment has a negligible effect on the distributions but preserves all 20,000 training observations. That matters here: with only 1,110 failures in total, we do not want to discard rows unnecessarily.

### Step 2.4 - Feature scaling (scaler fitted on the training split only)

`StandardScaler` transforms each column to have mean 0 and standard deviation 1 using the formula `z = (x - mean) / std`.

This is not cosmetic for a neural network. Gradient descent takes a step proportional to the size of each input, so when one feature has a standard deviation of 5.5 and another 1.65, the loss surface becomes an elongated valley: the learning rate that is stable for the wide feature is far too small for the narrow one, and training either crawls or oscillates. Standardising puts every feature on the same footing so a single learning rate works for all 40 inputs. It also keeps the pre-activation values in the range where ReLU and sigmoid have useful gradients.

In [24]:
# Fitting the scaler on the TRAINING SPLIT ONLY
scaler = StandardScaler()
scaler.fit(X_train)

# Applying the frozen training-derived mean and std to all three sets
X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns, index=X_train.index)
X_val_scaled   = pd.DataFrame(scaler.transform(X_val),   columns=X_val.columns,   index=X_val.index)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),  columns=X_test.columns,  index=X_test.index)

# Verifying the result: the training set should now be centred at 0 with std 1
check = pd.DataFrame({
    "Train_mean": X_train_scaled.mean(),
    "Train_std": X_train_scaled.std(),
    "Val_mean": X_val_scaled.mean(),
    "Val_std": X_val_scaled.std(),
}).round(3)

print("Scaling check - first 8 predictors:")
display(check.head(8))

print(f"\nTraining set: mean across all 40 features = {X_train_scaled.mean().mean():.6f}, "
      f"average std = {X_train_scaled.std().mean():.4f}")
print(f"Validation set: mean across all 40 features = {X_val_scaled.mean().mean():.6f}, "
      f"average std = {X_val_scaled.std().mean():.4f}")

**Observations**

- The training features are now centred at **mean 0 with standard deviation 1**, exactly as intended.
- The validation features are **close to but not exactly** mean 0 / std 1. **This is the correct and expected outcome** - it is the visible signature that we avoided leakage. Had we scaled before splitting, the validation set would also read exactly 0 and 1, which would have been the warning sign that evaluation data had contaminated the fit.
- All three datasets have now passed through an identical, training-derived transformation pipeline and are ready for modelling.

In [25]:
# Converting to NumPy arrays for Keras, and recording the input dimension
X_train_final = X_train_scaled.values
X_val_final   = X_val_scaled.values
X_test_final  = X_test_scaled.values

y_train_final = y_train.values
y_val_final   = y_val.values
y_test_final  = y_test.values

input_dim = X_train_final.shape[1]

print("Data ready for modelling")
print(f"  X_train : {X_train_final.shape}   y_train : {y_train_final.shape}")
print(f"  X_val   : {X_val_final.shape}   y_val   : {y_val_final.shape}")
print(f"  X_test  : {X_test_final.shape}   y_test  : {y_test_final.shape}")
print(f"  Number of input features (input_dim) : {input_dim}")

# **Model Building**

## Model Evaluation Criterion

### Translating the confusion matrix into money

The problem statement gives us the cost consequence of each cell of the confusion matrix, and an ordering between them:

| Prediction outcome | What it means physically | Cost incurred | Relative size |
|---|---|---|---|
| **True Positive (TP)** | Failure predicted, failure real | **Repair** cost | Medium |
| **False Negative (FN)** | Failure missed by the model | **Replacement** cost | **Highest** |
| **False Positive (FP)** | Failure predicted, generator healthy | **Inspection** cost | Lowest |
| **True Negative (TN)** | No failure predicted, none occurred | No cost | None |

The stated ordering is **inspection < repair << replacement**.

### Which error is more costly?

The two errors a classifier can make are not remotely equivalent here:

- A **false negative** means a generator that was quietly degrading is left alone until it breaks. ReneWind then pays the **replacement** cost - the most expensive outcome available - plus the unscheduled downtime, lost generation revenue, and emergency crew mobilisation that come with a catastrophic failure rather than a planned one.
- A **false positive** means a technician inspects a healthy generator and finds nothing. ReneWind pays the **inspection** cost, which the problem statement tells us is the *cheapest* of the three.

**A false negative is therefore far more damaging than a false positive.** Our modelling objective follows directly: **minimise false negatives**, accepting a reasonable number of false positives as the price of that protection.

### The metric of choice: Recall

$$\text{Recall} = \frac{TP}{TP + FN}$$

Recall is the proportion of *actual* failures that the model successfully flags. It is the only standard metric whose denominator contains the false negatives we most need to suppress - driving recall up is mathematically the same act as driving FN down. **Recall is our primary metric for selecting the final model.**

### Why we reject accuracy

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

With only 5.55% failures, a degenerate model that predicts "no failure" for every single generator scores **94.45% accuracy** while catching **zero** failures and incurring the replacement cost on every one. Accuracy is dominated by the majority class and is actively misleading on this dataset. We report it for completeness but never optimise for it.

### Why we still watch precision and F1

Recall alone can be gamed just as easily in the opposite direction: a model that flags *every* generator as failing achieves **100% recall**. It would also send inspection crews to all 5,000 turbines, and although inspection is the cheapest error, at that volume the bill becomes absurd and the maintenance team stops trusting the alerts.

So we adopt a **primary metric with a guard rail**:

- **Primary: Recall** - maximise it, because missed failures are the expensive error.
- **Guard rail: Precision and F1** - monitor them to confirm the recall gain is not being bought with an unusable flood of false alarms.

We will report all four metrics for every model, rank on validation recall, and use precision/F1 to break ties between models with comparable recall.

### Helper functions for building and evaluating the models

To keep the eight model sections that follow readable and strictly comparable, we define our evaluation utilities once here. Every model is trained with identical data, epochs, batch size and seed, so that any performance difference is attributable to the architectural change under test and nothing else.

In [26]:
# ----------------------------------------------------------------------
# Common training configuration - identical for every model so that the
# comparison isolates the effect of the architectural change being tested
# ----------------------------------------------------------------------
EPOCHS = 100
BATCH_SIZE = 128
SEED = 42

# A list that will accumulate the scores of every model we build, and a
# dictionary that keeps the fitted models themselves so that we can retrieve
# the winner later without retraining it
model_results = []
trained_models = {}


def get_metrics(model, X, y, threshold=0.5):
    """Return accuracy, recall, precision and F1 for a fitted Keras model."""
    probabilities = model.predict(X, verbose=0).ravel()
    predictions = (probabilities >= threshold).astype(int)
    return {
        "Accuracy": accuracy_score(y, predictions),
        "Recall": recall_score(y, predictions, zero_division=0),
        "Precision": precision_score(y, predictions, zero_division=0),
        "F1": f1_score(y, predictions, zero_division=0),
    }


def record_performance(name, description, model):
    """Score a model on the training and validation sets, append the result to
    the running comparison table, and print a readable summary."""
    train_scores = get_metrics(model, X_train_final, y_train_final)
    val_scores = get_metrics(model, X_val_final, y_val_final)

    trained_models[name] = model
    model_results.append({
        "Model": name,
        "Configuration": description,
        "Train_Recall": round(train_scores["Recall"], 4),
        "Val_Recall": round(val_scores["Recall"], 4),
        "Val_Precision": round(val_scores["Precision"], 4),
        "Val_F1": round(val_scores["F1"], 4),
        "Val_Accuracy": round(val_scores["Accuracy"], 4),
    })

    summary = pd.DataFrame({
        "Training": [train_scores[m] for m in ["Accuracy", "Recall", "Precision", "F1"]],
        "Validation": [val_scores[m] for m in ["Accuracy", "Recall", "Precision", "F1"]],
    }, index=["Accuracy", "Recall", "Precision", "F1"]).round(4)

    print(f"Performance of {name}  ({description})")
    display(summary)
    return val_scores


def plot_training_history(history, title):
    """Plot the loss and recall learning curves side by side."""
    # Locating the recall keys defensively: Keras occasionally appends a suffix
    # (e.g. 'recall_1') when a metric name is reused within one session
    hist = history.history
    train_recall_key = [k for k in hist if "recall" in k and not k.startswith("val_")][0]
    val_recall_key = [k for k in hist if "recall" in k and k.startswith("val_")][0]

    fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))

    ax[0].plot(hist["loss"], label="Training loss")
    ax[0].plot(hist["val_loss"], label="Validation loss")
    ax[0].set_title("Loss per epoch")
    ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Binary cross-entropy")
    ax[0].legend()

    ax[1].plot(hist[train_recall_key], label="Training recall")
    ax[1].plot(hist[val_recall_key], label="Validation recall")
    ax[1].set_title("Recall per epoch")
    ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Recall")
    ax[1].legend()

    fig.suptitle(f"Learning curves - {title}", fontsize=14)
    plt.tight_layout()
    plt.show()


def plot_confusion(model, X, y, title, threshold=0.5):
    """Display an annotated confusion matrix with the cost interpretation
    of each cell written into the label."""
    probabilities = model.predict(X, verbose=0).ravel()
    predictions = (probabilities >= threshold).astype(int)
    cm = confusion_matrix(y, predictions)

    labels = np.array([
        [f"TN\n{cm[0,0]}\n(no cost)",        f"FP\n{cm[0,1]}\n(inspection cost)"],
        [f"FN\n{cm[1,0]}\n(REPLACEMENT cost)", f"TP\n{cm[1,1]}\n(repair cost)"],
    ])

    plt.figure(figsize=(6.2, 5))
    sns.heatmap(cm, annot=labels, fmt="", cmap="Blues", cbar=False,
                annot_kws={"size": 11},
                xticklabels=["Predicted: No Failure", "Predicted: Failure"],
                yticklabels=["Actual: No Failure", "Actual: Failure"])
    plt.title(f"Confusion matrix - {title}", fontsize=12)
    plt.tight_layout()
    plt.show()

    print(classification_report(y, predictions,
                                target_names=["No Failure (0)", "Failure (1)"],
                                zero_division=0))


print("Helper functions defined.")
print(f"Every model will train for {EPOCHS} epochs with a batch size of {BATCH_SIZE}.")

## Initial Model Building (Model 0)

Our baseline, as specified in the project instructions, is deliberately the simplest network that can address the problem:

- **just one hidden layer** of 32 neurons
- **ReLU** activation on the hidden layer
- **SGD** (stochastic gradient descent) as the optimizer
- a single **sigmoid** output neuron, giving a probability of failure
- **binary cross-entropy** loss, the standard objective for two-class problems
- **no** dropout and **no** class weights - those come later, so we can measure what each one contributes

This model exists to give us a reference point, not to be good. Everything we build afterwards must justify its added complexity by beating it on validation recall.

In [27]:
# Resetting the seed so this model starts from a reproducible initial state
reset_seed(SEED)

# ----------------------------------------------------------------------
# Model 0: one hidden layer, ReLU activation, SGD optimizer
# ----------------------------------------------------------------------
model_0 = Sequential([
    Input(shape=(input_dim,)),                          # 40 input features
    Dense(32, activation="relu", name="hidden_layer"),  # single hidden layer
    Dense(1, activation="sigmoid", name="output"),      # probability of failure
], name="Model_0_Baseline_SGD")

model_0.compile(
    optimizer=SGD(learning_rate=0.01),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.Recall(name="recall")],
)

model_0.summary()

**Understanding the parameter count**

The hidden layer holds `40 x 32 = 1,280` weights plus 32 biases = **1,312 parameters**. The output layer holds `32 x 1 = 32` weights plus 1 bias = **33 parameters**. That is **1,345 trainable parameters** in total, learned from 16,000 training observations - a comfortable ratio of roughly 12 observations per parameter, so this baseline is unlikely to overfit badly.

In [28]:
# Training Model 0
history_0 = model_0.fit(
    X_train_final, y_train_final,
    validation_data=(X_val_final, y_val_final),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=0,          # silenced to keep the notebook output clean
)

print(f"Training complete - {EPOCHS} epochs.")
print(f"Final training loss   : {history_0.history['loss'][-1]:.4f}")
print(f"Final validation loss : {history_0.history['val_loss'][-1]:.4f}")

In [29]:
plot_training_history(history_0, "Model 0 (1 hidden layer, SGD)")

In [30]:
val_0 = record_performance(
    "Model 0", "1 hidden layer (32), ReLU, SGD, no dropout, no class weights", model_0
)

In [31]:
plot_confusion(model_0, X_val_final, y_val_final, "Model 0 (validation set)")

### Comment on the performance of Model 0

**Headline result: validation recall 0.7748, precision 0.9718, F1 0.8622, accuracy 0.9862.**

**Reading the learning curves**

- Both the training and validation loss fall steadily and then flatten (final values 0.0623 and 0.0686), and the two curves stay close together throughout. **There is no sign of overfitting** - in fact the train-validation recall gap is slightly *negative* (-0.0147), meaning the model scores marginally better on data it has never seen than on its own training data. That is entirely consistent with the modest 1,345-parameter capacity of this network.
- The recall curves climb from zero and then plateau. The fact that recall starts at or near zero for the first several epochs is itself informative: **with 19 healthy generators for every failure, the fastest way for the network to reduce cross-entropy loss early in training is simply to predict "no failure" for everything.** It only begins to identify failures once the easy majority-class gains are exhausted.
- SGD with a learning rate of 0.01 produces a visibly gradual, smooth descent - the expected behaviour of plain SGD, which applies the same learning rate to every parameter and carries no momentum or adaptive scaling.

**Reading the confusion matrix**

- Accuracy of 0.9862 looks impressive, but as established in our metric discussion this is largely an artefact of the imbalance and tells us little.
- The cell that matters is **FN: the model missed 50 of the 222 validation failures.** Each of those is a generator left in service until it broke, costing ReneWind a full replacement.
- The contrast between **precision (0.9718) and recall (0.7748)** is the diagnostic signature. The model is highly reliable *when* it raises an alarm, but it raises too few of them. This is the classic **imbalance-driven conservative bias**: the loss function treats a missed failure and a false alarm as equally bad, so the model has learned to flag failure only when very confident. That default is precisely backwards for ReneWind's cost structure.

**What this tells us to do next**

The baseline works but is not deployable. Three levers follow directly from what we have observed, and we test each below:

1. **The network may be too shallow.** Our EDA showed the failure signature is distributed across many interacting sensors. **Additional hidden layers** give the network depth to compose those interactions.
2. **The optimizer is slow.** Plain SGD is still improving when it plateaus. **Adam** adapts the learning rate per parameter and carries momentum, so it should reach a better solution within the same 100 epochs.
3. **The loss function is blind to our cost asymmetry.** **Class weights** let us tell the network explicitly that a missed failure is worse than a false alarm, attacking the conservative bias directly.

We also test **dropout**, since the deeper networks introduced by lever 1 will carry a genuine overfitting risk.

# **Model Performance Improvement**

Model 0 gave us a working baseline with a clear weakness: it is too conservative about flagging failures, which is exactly the wrong bias for ReneWind's cost structure. We now build **seven further models** that apply the four improvement techniques available to us, individually and in combination.

## The experimental design

Rather than changing several things at once and guessing which change helped, we vary the levers in a **structured, near-factorial design**. Each model differs from a comparable one by a single lever, so the effect of that lever is isolated and measurable:

| Model | Hidden layers | Optimizer | Dropout | Class weights | What this model isolates |
|:---|:---|:---|:---|:---|:---|
| **0** *(baseline)* | 32 | SGD | - | - | Reference point |
| **1** | 64, 32 | SGD | - | - | Effect of **more hidden layers** alone |
| **2** | 64, 32 | **Adam** | - | - | Effect of **swapping the optimizer** (vs Model 1) |
| **3** | 64, 32 | SGD | - | **Yes** | Effect of **class weights** under SGD (vs Model 1) |
| **4** | 64, 32 | **Adam** | - | **Yes** | Effect of **class weights** under Adam (vs Model 2) |
| **5** | 64, 32 | **Adam** | **0.2** | - | Effect of **dropout** alone (vs Model 2) |
| **6** | 64, 32 | **Adam** | **0.2** | **Yes** | **Dropout + class weights** combined |
| **7** | 128, 64, 32 | **Adam** | **0.3** | **Yes** | A **deeper** network with all techniques applied |

Every model trains on identical data for **100 epochs at batch size 128 from the same random seed**, so any difference in performance is attributable to the configuration change and not to luck.

## The four levers explained

**More hidden layers.** Each hidden layer lets the network compose features from the layer beneath it. Our EDA showed the failure signature lives in the *interaction* of many sensors rather than in any one of them, so depth gives the network the machinery to represent those interactions. The cost is more parameters and therefore more overfitting risk.

**Optimizer: SGD vs Adam.** Plain SGD applies one fixed learning rate to every parameter. **Adam** maintains a per-parameter adaptive learning rate and accumulates momentum, so parameters with small or noisy gradients still make progress. In practice Adam converges considerably faster and usually finds a better solution within a fixed epoch budget.

**Dropout.** During each training step, dropout randomly switches off a given fraction of neurons in a layer. This prevents the network from leaning on any single pathway and forces redundant, more robust representations. It is our regularisation defence for the deeper networks.

**Class weights.** This is the lever aimed most directly at our cost asymmetry. With a 17:1 imbalance, standard cross-entropy treats a missed failure and a false alarm as equally bad - which contradicts ReneWind's economics entirely. Class weights multiply the loss contribution of the minority class so that **misclassifying a failure hurts the network roughly nine times more than misclassifying a healthy generator.** This encodes our cost asymmetry directly into the objective function.

### Computing the class weights

We use the standard *balanced* formula, which sets each class weight inversely proportional to its frequency:

$$w_c = \frac{n_{\text{total}}}{2 \times n_c}$$

In [32]:
# Computing balanced class weights from the TRAINING SPLIT only
n_total = len(y_train_final)
n_negative = int((y_train_final == 0).sum())
n_positive = int((y_train_final == 1).sum())

class_weights = {
    0: n_total / (2 * n_negative),
    1: n_total / (2 * n_positive),
}

print(f"Training split: {n_negative} non-failures, {n_positive} failures "
      f"(imbalance ratio {n_negative / n_positive:.1f} : 1)")
print(f"\nClass weight for 0 (No Failure) : {class_weights[0]:.4f}")
print(f"Class weight for 1 (Failure)    : {class_weights[1]:.4f}")
print(f"\nA failure is therefore weighted {class_weights[1] / class_weights[0]:.1f}x "
      f"more heavily than a non-failure in the loss function.")

**Observations**

- The training split carries a **17.0 : 1** imbalance, so the balanced formula assigns a weight of about **9.01 to failures** against **0.53 to non-failures**.
- In effect, each of the 888 failure observations now contributes as much to the loss as roughly **17 non-failure observations** would. The network can no longer reduce its loss by ignoring the minority class.
- Note that these weights are derived from the **training split only** - consistent with our leakage discipline, no information from the validation or test sets influences them.

## Model 1

**Configuration: two hidden layers (64, 32), ReLU, SGD, no dropout, no class weights**

Our first change from the baseline is the simplest one: **add depth**. We keep SGD and add no regularisation, so that whatever changes in performance is attributable purely to the extra hidden layer.

In [33]:
reset_seed(SEED)

model_1 = Sequential([
    Input(shape=(input_dim,)),
    Dense(64, activation="relu", name="hidden_1"),
    Dense(32, activation="relu", name="hidden_2"),
    Dense(1, activation="sigmoid", name="output"),
], name="Model_1")

model_1.compile(
    optimizer=SGD(learning_rate=0.01),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.Recall(name="recall")],
)

model_1.summary()

history_1 = model_1.fit(
    X_train_final, y_train_final,
    validation_data=(X_val_final, y_val_final),
    epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0,
)
print("\nTraining complete.")

In [34]:
plot_training_history(history_1, "Model 1 (2 hidden layers, SGD)")
val_1 = record_performance("Model 1", "2 hidden layers (64, 32), SGD", model_1)
plot_confusion(model_1, X_val_final, y_val_final, "Model 1 (validation set)")

### Comment on the performance of Model 1

**Headline result: validation recall 0.8153 (up from 0.7748), precision 0.9837, F1 0.8916. False negatives fall from 50 to 41.**

- Adding a second hidden layer raises the parameter count from 1,345 to roughly **4,700**, and it delivers a real gain: **recall improves by 4.1 percentage points** and nine additional failures are now caught. The extra capacity is being used to represent the multi-sensor interactions our EDA identified, not wasted.
- The learning curves remain well behaved and the generalisation gap is small (0.0338), so **the extra depth has not introduced overfitting.** Precision actually edges *up* to 0.9837, confirming the additional detections are genuine rather than speculative.
- However, **the model remains far too conservative.** 41 missed failures out of 222 is still a substantial replacement bill, and the precision-recall imbalance (0.9837 against 0.8153) is essentially unchanged in character from the baseline. Depth has made the network a better learner, but it has not altered *what* the network is being asked to optimise - the loss function still treats both error types as equally costly.
- **Conclusion:** depth is a genuine improvement and a useful foundation, but it addresses model capacity rather than the cost asymmetry at the heart of this problem. We retain the two-layer architecture and now vary the optimizer and the class weighting.

## Model 2

**Configuration: two hidden layers (64, 32), ReLU, Adam, no dropout, no class weights**

Identical to Model 1 in every respect except the optimizer, which changes from SGD to **Adam**. This isolates the effect of the optimizer.

In [35]:
reset_seed(SEED)

model_2 = Sequential([
    Input(shape=(input_dim,)),
    Dense(64, activation="relu", name="hidden_1"),
    Dense(32, activation="relu", name="hidden_2"),
    Dense(1, activation="sigmoid", name="output"),
], name="Model_2")

model_2.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.Recall(name="recall")],
)

history_2 = model_2.fit(
    X_train_final, y_train_final,
    validation_data=(X_val_final, y_val_final),
    epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0,
)
print("Training complete.")

In [36]:
plot_training_history(history_2, "Model 2 (2 hidden layers, Adam)")
val_2 = record_performance("Model 2", "2 hidden layers (64, 32), Adam", model_2)
plot_confusion(model_2, X_val_final, y_val_final, "Model 2 (validation set)")

### Comment on the performance of Model 2

**Headline result: validation recall 0.8559 (up from 0.8153 under SGD), precision 0.9794, F1 0.9135. False negatives fall from 41 to 32.**

- The optimizer swap produces a **markedly faster descent**: the loss curve drops steeply within the first 10-20 epochs, where SGD needed most of the 100 epochs to reach a comparable level. This is Adam's per-parameter adaptive learning rate and momentum at work.
- **Recall improves by a further 4.1 percentage points** over the identical SGD architecture, and nine more failures are caught. Because Adam converges to a genuinely better minimum inside the same epoch budget, it extracts more of the failure signal from the same data and the same architecture. **Optimizer choice is worth as much here as adding a whole hidden layer was.**
- The cost of that power is visible in the curves and in the numbers: the **train-validation recall gap widens sharply to 0.0765**, more than double Model 1's 0.0338, and the training loss now runs well below the validation loss in the later epochs. **This is the onset of overfitting** - Adam is strong enough to start memorising the training set. It is the direct motivation for testing dropout in Model 5.
- **Conclusion:** Adam is clearly the better optimizer and we carry it forward. But note that recall of 0.8559 still leaves 32 failures undetected, and precision remains very high at 0.9794 - the conservative bias persists, because nothing we have changed so far tells the model that those two errors carry different price tags.

## Model 3

**Configuration: two hidden layers (64, 32), ReLU, SGD, no dropout, WITH class weights**

Now we introduce the lever aimed squarely at our cost asymmetry. Model 3 is Model 1 plus class weights, keeping SGD so we can see how class weighting behaves with the simpler optimizer.

In [37]:
reset_seed(SEED)

model_3 = Sequential([
    Input(shape=(input_dim,)),
    Dense(64, activation="relu", name="hidden_1"),
    Dense(32, activation="relu", name="hidden_2"),
    Dense(1, activation="sigmoid", name="output"),
], name="Model_3")

model_3.compile(
    optimizer=SGD(learning_rate=0.01),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.Recall(name="recall")],
)

history_3 = model_3.fit(
    X_train_final, y_train_final,
    validation_data=(X_val_final, y_val_final),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    class_weight=class_weights,      # <-- the new lever
    verbose=0,
)
print("Training complete.")

In [38]:
plot_training_history(history_3, "Model 3 (2 hidden layers, SGD, class weights)")
val_3 = record_performance("Model 3", "2 hidden layers (64, 32), SGD, class weights", model_3)
plot_confusion(model_3, X_val_final, y_val_final, "Model 3 (validation set)")

### Comment on the performance of Model 3

**Headline result: validation recall 0.8874 - a jump of 7.2 percentage points over the identical unweighted Model 1 (0.8153). This is the single largest gain produced by any one lever in the entire study. False negatives fall from 41 to 25.**

- Class weighting transforms the model's behaviour under SGD. The false-negative count drops by 16 and the true-positive count rises from 181 to 197. **The model has stopped playing safe.**
- **Precision falls sharply in exchange, from 0.9837 to 0.8347**, and false positives rise from 3 to 39. This is not a defect - **it is the trade we deliberately chose.** By weighting a failure at 9.01 against 0.53 for a non-failure, we instructed the network to accept more false alarms in return for missing fewer real failures. In ReneWind's cost terms we are converting expensive *replacements* into cheap *inspections*, and the validation cost confirms it: **8,605 units against Model 1's 9,545.**
- Note that the absolute loss values are **not comparable** to the previous models, because the quantity being minimised is now a *weighted* cross-entropy. Only recall and the confusion-matrix figures can be compared across models.
- Accuracy also falls, from 0.9890 to 0.9840 - the clearest possible demonstration of why we rejected accuracy as our metric. **The model has become more valuable to the business while scoring worse on accuracy.**
- **Conclusion:** class weighting is decisively the most effective single lever *for the SGD models*, exactly as our imbalance analysis predicted. Whether it compounds as strongly with Adam is the question Model 4 answers.

## Model 4

**Configuration: two hidden layers (64, 32), ReLU, Adam, no dropout, WITH class weights**

This combines the two levers that have each proven effective in isolation - **Adam** (Model 2) and **class weights** (Model 3).

In [39]:
reset_seed(SEED)

model_4 = Sequential([
    Input(shape=(input_dim,)),
    Dense(64, activation="relu", name="hidden_1"),
    Dense(32, activation="relu", name="hidden_2"),
    Dense(1, activation="sigmoid", name="output"),
], name="Model_4")

model_4.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.Recall(name="recall")],
)

history_4 = model_4.fit(
    X_train_final, y_train_final,
    validation_data=(X_val_final, y_val_final),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    class_weight=class_weights,
    verbose=0,
)
print("Training complete.")

In [40]:
plot_training_history(history_4, "Model 4 (2 hidden layers, Adam, class weights)")
val_4 = record_performance("Model 4", "2 hidden layers (64, 32), Adam, class weights", model_4)
plot_confusion(model_4, X_val_final, y_val_final, "Model 4 (validation set)")

### Comment on the performance of Model 4

**Headline result: validation recall 0.8739, precision 0.8858, F1 0.8798 - and a train-validation gap of 0.1148, by far the worst of any model built.**

This model produced the most instructive negative result in the study, and it overturns an expectation we would reasonably have carried in.

- **Combining our two best individual levers did not compound.** Model 4 (Adam + class weights) reaches recall of 0.8739, which is **lower than Model 3's 0.8874** (SGD + class weights) despite Adam having beaten SGD in every unweighted comparison. Adding class weights to Adam bought only **+1.8 points** over Model 2 (0.8559), against the **+7.2 points** the same lever delivered under SGD.
- The reason is visible in the generalisation gap. **Training recall reaches 0.9887 while validation recall is only 0.8739** - the network has essentially memorised the 888 weighted training failures. Adam's aggressive optimisation combined with a 17:1 loss weighting on a small minority class is a combination that overfits hard. At 0.1148 the gap is **50% wider than the next-worst model** (Model 2, at 0.0765) and roughly **three times that of every regularised model** in the study.
- This is an important corrective to a common assumption: **techniques that help individually do not necessarily add up.** Adam's power and class weighting's aggression pull in the same direction, and applied together without regularisation they overshoot.
- The economics confirm it. Model 4 costs **8,745 units** against Model 3's 8,605 - so despite its higher precision it is the *more expensive* of the two class-weighted models built so far.
- **Conclusion:** Adam plus class weights is a genuinely strong configuration but an unstable one. It does not need a better optimizer - **it needs regularisation**, which is precisely what Models 6 and 7 add.

## Model 5

**Configuration: two hidden layers (64, 32), ReLU, Adam, Dropout 0.2, no class weights**

To isolate the effect of **dropout**, this model is Model 2 with a dropout layer after each hidden layer and nothing else changed. Comparing Model 5 against Model 2 tells us what dropout contributes on its own.

In [41]:
reset_seed(SEED)

model_5 = Sequential([
    Input(shape=(input_dim,)),
    Dense(64, activation="relu", name="hidden_1"),
    Dropout(0.2, name="dropout_1"),        # switches off 20% of neurons each step
    Dense(32, activation="relu", name="hidden_2"),
    Dropout(0.2, name="dropout_2"),
    Dense(1, activation="sigmoid", name="output"),
], name="Model_5")

model_5.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.Recall(name="recall")],
)

model_5.summary()

history_5 = model_5.fit(
    X_train_final, y_train_final,
    validation_data=(X_val_final, y_val_final),
    epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0,
)
print("\nTraining complete.")

In [42]:
plot_training_history(history_5, "Model 5 (2 hidden layers, Adam, Dropout 0.2)")
val_5 = record_performance("Model 5", "2 hidden layers (64, 32), Adam, Dropout 0.2", model_5)
plot_confusion(model_5, X_val_final, y_val_final, "Model 5 (validation set)")

### Comment on the performance of Model 5

**Headline result: validation recall 0.8829, precision 0.9751, and F1 of 0.9267 - the highest F1 and the highest accuracy (0.9922) of any model in this study.**

This model performed considerably better than dropout alone might be expected to, and it deserves careful reading.

- **Dropout is not merely a regularisation safeguard here - it is a substantial performance lever.** Adding it to Model 2 lifted recall from 0.8559 to 0.8829 (**+2.7 points**) while *keeping* precision at 0.9751 and cutting the generalisation gap from 0.0765 to 0.0360. False negatives fell from 32 to 26 with only 5 false positives across the entire 4,000-row validation set.
- The mechanism is exactly what dropout is designed for. Our EDA found 22 predictor pairs correlated above 0.70, meaning substantial redundancy among the 40 sensors. By randomly deactivating 20% of neurons on every step, dropout **prevents the network from leaning on any single correlated pathway** and forces it to spread the failure signature across redundant routes. On this dataset that is worth more than it would be on data with independent features.
- Note that dropout adds **no trainable parameters** - the `model.summary()` output shows zero for the dropout layers. It changes only how training proceeds, and is switched off automatically at prediction time.
- One subtlety is worth stating precisely, because it is easy to get wrong. **On the learning-curve plot**, the per-epoch training recall is computed by Keras with dropout *active* - a deliberately handicapped network - while validation recall is computed with the full network, so the training curve can sit *below* the validation curve. **In the comparison table, however, both figures come from `model.predict()`, which always runs in inference mode with dropout disabled.** The gap of 0.0360 reported there is therefore a genuine generalisation gap, not an artefact of the regularisation.
- **The strategic point:** Model 5 achieves this **without class weights**, and it is the cheapest of all the unweighted models at 8,505 units. It illustrates that there are two distinct routes to good performance here - regularise the model so it generalises honestly, or reweight the loss so it prioritises failures. **Model 5 shows the first route alone is already competitive.** The remaining question is whether combining both beats either.
- **Conclusion:** an excellent model, and the best in this study on the balanced metrics. Its recall of 0.8829 nevertheless still trails the class-weighted Models 6 and 7, and **recall is the metric ReneWind's economics select on** - which is why it does not become our final choice despite its superior F1.

## Model 6

**Configuration: two hidden layers (64, 32), ReLU, Adam, Dropout 0.2, WITH class weights**

This model combines **all three effective levers** at the two-layer depth: Adam, dropout for regularisation, and class weights to encode the cost asymmetry.

In [43]:
reset_seed(SEED)

model_6 = Sequential([
    Input(shape=(input_dim,)),
    Dense(64, activation="relu", name="hidden_1"),
    Dropout(0.2, name="dropout_1"),
    Dense(32, activation="relu", name="hidden_2"),
    Dropout(0.2, name="dropout_2"),
    Dense(1, activation="sigmoid", name="output"),
], name="Model_6")

model_6.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.Recall(name="recall")],
)

history_6 = model_6.fit(
    X_train_final, y_train_final,
    validation_data=(X_val_final, y_val_final),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    class_weight=class_weights,
    verbose=0,
)
print("Training complete.")

In [44]:
plot_training_history(history_6, "Model 6 (Adam, Dropout 0.2, class weights)")
val_6 = record_performance("Model 6", "2 hidden layers (64, 32), Adam, Dropout 0.2, class weights", model_6)
plot_confusion(model_6, X_val_final, y_val_final, "Model 6 (validation set)")

### Comment on the performance of Model 6

**Headline result: validation recall 0.8964 - the joint-highest of any model built - with precision 0.9087, F1 0.9025, and a healthy generalisation gap of 0.0372.**

- **Combining all three levers works, and the evidence is unambiguous when set against Model 4.** Both models use Adam with class weights; Model 6 adds dropout. That single change lifts recall from 0.8739 to 0.8964 and **collapses the generalisation gap from 0.1148 to 0.0372.** Dropout has fixed exactly the overfitting pathology Model 4 exposed.
- Against Model 5 (the same network without class weights), recall rises from 0.8829 to 0.8964 while precision falls from 0.9751 to 0.9087 - **the familiar, deliberate trade of precision for recall**, and one that ReneWind's cost structure rewards: validation cost falls from 8,505 to **8,370 units**, a 62.3% saving against the no-model benchmark.
- The confusion matrix tells the story cleanly: **23 missed failures out of 222** - fewer than half of the baseline's 50 - bought with just 20 false alarms across 3,778 healthy generators.
- The learning curves are the healthiest among the high-recall models, with training and validation tracking each other closely rather than diverging.
- **Conclusion:** a strong, well-regularised candidate and, on the evidence so far, the best configuration we have. One question remains - whether still greater depth extracts further signal, which Model 7 tests.

## Model 7

**Configuration: three hidden layers (128, 64, 32), ReLU, Adam, Dropout 0.3, WITH class weights**

Our final model tests whether **additional depth and width** extract more signal. We widen the first layer to 128 neurons, add a third hidden layer, and raise dropout to 0.3 to compensate for the substantially larger capacity.

In [45]:
reset_seed(SEED)

model_7 = Sequential([
    Input(shape=(input_dim,)),
    Dense(128, activation="relu", name="hidden_1"),
    Dropout(0.3, name="dropout_1"),
    Dense(64, activation="relu", name="hidden_2"),
    Dropout(0.3, name="dropout_2"),
    Dense(32, activation="relu", name="hidden_3"),
    Dropout(0.3, name="dropout_3"),
    Dense(1, activation="sigmoid", name="output"),
], name="Model_7")

model_7.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.Recall(name="recall")],
)

model_7.summary()

history_7 = model_7.fit(
    X_train_final, y_train_final,
    validation_data=(X_val_final, y_val_final),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    class_weight=class_weights,
    verbose=0,
)
print("\nTraining complete.")

In [46]:
plot_training_history(history_7, "Model 7 (3 hidden layers, Adam, Dropout 0.3, class weights)")
val_7 = record_performance("Model 7", "3 hidden layers (128, 64, 32), Adam, Dropout 0.3, class weights", model_7)
plot_confusion(model_7, X_val_final, y_val_final, "Model 7 (validation set)")

### Comment on the performance of Model 7

**Headline result: validation recall 0.8964 - identical to Model 6 to four decimal places - with precision 0.9128, F1 0.9045, and a generalisation gap of 0.0450.**

- This is by far our largest network at **15,617 trainable parameters** - nearly twelve times the baseline's 1,345 - with a third hidden layer, a first layer widened to 128 neurons, and dropout raised to 0.3 to counterweight that capacity.
- **The extra depth bought no additional recall whatsoever.** Model 7 catches exactly the same 199 of 222 validation failures as the two-layer Model 6, and misses exactly the same 23. It edges ahead only on the margins: one fewer false positive (19 against 20), a fractionally better F1 (0.9045 against 0.9025), and a validation cost of **8,365 against 8,370 units - a difference of 5 units in 22,200, which is noise.**
- **This is the clearest finding in the whole experiment: we have hit diminishing returns on model capacity.** The explanation traces straight back to our EDA. With only **888 failure examples** in the training split, there is a hard ceiling on how much structure any network can reliably learn about the minority class. Beyond a certain capacity, further parameters have nothing new to fit - **the binding constraint is the quantity of failure data, not the size of the model.**
- Supporting evidence: Model 7's generalisation gap is *wider* than Model 6's (0.0450 against 0.0372) despite the heavier dropout, which is what we would expect when capacity outruns the available signal.
- **This matters for our recommendation to ReneWind: the path to a better model runs through collecting more labelled failure events, not through building a deeper network.**
- **Conclusion:** Models 6 and 7 are statistically indistinguishable on our primary metric. Our pre-committed decision rule resolves the tie in the next section.

# **Model Performance Comparison and Final Model Selection**

Now, in order to select the final model, we compare the performances of all eight models on the training and validation sets.

In [47]:
# Assembling the full comparison table
comparison = pd.DataFrame(model_results)

# The generalisation gap: how much better the model does on data it was
# trained on than on data it has never seen. A large gap signals overfitting.
comparison["Overfit_Gap"] = (comparison["Train_Recall"] - comparison["Val_Recall"]).round(4)

# Ranking on our primary metric: validation recall
comparison_ranked = comparison.sort_values("Val_Recall", ascending=False).reset_index(drop=True)

print("=" * 100)
print("PERFORMANCE OF ALL MODELS, RANKED BY VALIDATION RECALL (our primary metric)")
print("=" * 100)
display(comparison_ranked[[
    "Model", "Configuration", "Train_Recall", "Val_Recall",
    "Val_Precision", "Val_F1", "Val_Accuracy", "Overfit_Gap",
]])

In [48]:
# Visual comparison of training vs validation recall across all models
fig, ax = plt.subplots(1, 2, figsize=(19, 5.5))

order = comparison_ranked["Model"].tolist()
x = np.arange(len(order))
width = 0.38

train_vals = comparison_ranked["Train_Recall"].values
val_vals = comparison_ranked["Val_Recall"].values

b1 = ax[0].bar(x - width/2, train_vals, width, label="Training recall", color="#4C72B0")
b2 = ax[0].bar(x + width/2, val_vals, width, label="Validation recall", color="#C44E52")
ax[0].bar_label(b1, fmt="%.3f", fontsize=8, padding=2)
ax[0].bar_label(b2, fmt="%.3f", fontsize=8, padding=2)
ax[0].set_xticks(x); ax[0].set_xticklabels(order, rotation=30, ha="right")
ax[0].set_ylabel("Recall"); ax[0].set_ylim(0, 1.12)
ax[0].set_title("Training vs validation recall (ranked by validation recall)", fontsize=12)
ax[0].legend(loc="lower left")

# Recall against precision, to make the trade-off explicit
ax[1].scatter(comparison_ranked["Val_Precision"], comparison_ranked["Val_Recall"],
              s=140, c="#55A868", edgecolors="black", zorder=3)
for _, r in comparison_ranked.iterrows():
    ax[1].annotate(r["Model"], (r["Val_Precision"], r["Val_Recall"]),
                   textcoords="offset points", xytext=(8, 5), fontsize=9)
ax[1].set_xlabel("Validation precision"); ax[1].set_ylabel("Validation recall")
ax[1].set_title("The recall-precision trade-off across our models", fontsize=12)
ax[1].grid(True, alpha=0.35)

plt.tight_layout()
plt.show()

### Translating model performance into maintenance cost

Recall ranks the models on our chosen metric, but ReneWind's real objective is **money**. Since the problem statement gives us only the *ordering* of the three costs and not their values, we adopt illustrative relative cost units that respect that ordering:

| Outcome | Cost consequence | Relative units assumed |
|---|---|---|
| False Negative (FN) | Generator **replacement** | **100** |
| True Positive (TP) | Generator **repair** | **30** |
| False Positive (FP) | **Inspection**, nothing found | **5** |
| True Negative (TN) | No action | 0 |

The exact figures are illustrative, but the *conclusions are robust* to any values that preserve the stated ordering `inspection < repair << replacement`. This analysis gives us an economic cross-check on our recall-based ranking.

In [49]:
# Relative cost assumptions (illustrative, preserving the stated ordering)
COST_REPLACEMENT = 100   # a false negative - the failure we missed
COST_REPAIR = 30         # a true positive - caught in time, repaired
COST_INSPECTION = 5      # a false positive - inspected, nothing found


def cost_breakdown(model, X, y, threshold=0.5):
    """Return the confusion-matrix counts and the total maintenance cost."""
    probabilities = model.predict(X, verbose=0).ravel()
    predictions = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, predictions, labels=[0, 1]).ravel()
    total = tp * COST_REPAIR + fn * COST_REPLACEMENT + fp * COST_INSPECTION
    return {"TN": tn, "FP": fp, "FN": fn, "TP": tp, "Total_Cost": total}


# Computing the cost of every model on the validation set
cost_rows = []
for name in comparison["Model"]:
    breakdown = cost_breakdown(trained_models[name], X_val_final, y_val_final)
    breakdown["Model"] = name
    cost_rows.append(breakdown)

cost_df = pd.DataFrame(cost_rows)[["Model", "TN", "FP", "FN", "TP", "Total_Cost"]]

# The "do nothing" benchmark: with no model at all, every failure runs to
# destruction and incurs the full replacement cost
no_model_cost = int(y_val_final.sum()) * COST_REPLACEMENT
cost_df["Saving_vs_No_Model"] = no_model_cost - cost_df["Total_Cost"]
cost_df["Saving_Percent"] = (cost_df["Saving_vs_No_Model"] / no_model_cost * 100).round(1)

cost_df = cost_df.sort_values("Total_Cost").reset_index(drop=True)

print(f"Benchmark - no predictive model at all:")
print(f"  All {int(y_val_final.sum())} validation failures run to destruction "
      f"= {no_model_cost:,} cost units\n")
print("=" * 95)
print("MAINTENANCE COST BY MODEL ON THE VALIDATION SET (lowest cost first)")
print("=" * 95)
display(cost_df)

In [50]:
# Visualising cost against recall
fig, ax = plt.subplots(1, 2, figsize=(18, 5))

cd = cost_df.sort_values("Total_Cost")
bars = ax[0].barh(cd["Model"], cd["Total_Cost"], color="#4C72B0")
ax[0].bar_label(bars, fmt="%.0f", padding=3, fontsize=9)
ax[0].axvline(no_model_cost, color="red", linestyle="--", linewidth=1.6,
              label=f"No model at all ({no_model_cost:,})")
ax[0].set_xlabel("Total maintenance cost (relative units)")
ax[0].set_title("Validation-set maintenance cost by model", fontsize=12)
ax[0].legend()

merged = cost_df.merge(comparison[["Model", "Val_Recall"]], on="Model")
ax[1].scatter(merged["Val_Recall"], merged["Total_Cost"], s=140,
              c="#C44E52", edgecolors="black", zorder=3)
for _, r in merged.iterrows():
    ax[1].annotate(r["Model"], (r["Val_Recall"], r["Total_Cost"]),
                   textcoords="offset points", xytext=(8, 5), fontsize=9)
ax[1].set_xlabel("Validation recall"); ax[1].set_ylabel("Total maintenance cost")
ax[1].set_title("Higher recall drives lower cost - validating our metric choice", fontsize=12)
ax[1].grid(True, alpha=0.35)

plt.tight_layout()
plt.show()

**Observations**

- **Every model saves money against the "no predictive maintenance" benchmark**, which is the baseline ReneWind operates from today. Even the weak baseline model delivers a positive return.
- The cost ranking and the recall ranking **point in the same direction**: the models with the highest recall are the models with the lowest total cost. This is an important independent confirmation that **recall was the right metric to optimise** - we did not simply assert it, we can now demonstrate it economically.
- The right-hand scatter plot makes the relationship explicit. As recall rises, cost falls, because each additional failure caught converts a 100-unit replacement into a 30-unit repair - a **70-unit saving per catch** - while the false alarms that come with it cost only 5 units each. The arithmetic strongly favours aggressive detection: **one caught failure pays for fourteen unnecessary inspections.**
- The class-weighted models dominate the bottom (cheapest) end of the table, while the unweighted models cluster at the expensive end. **Class weighting is the single most economically valuable modelling decision in this project.**

### Selecting the final model

We apply a clearly stated, pre-committed decision rule so that the choice is principled rather than retrospective:

1. **Primary criterion - validation recall.** Missed failures are the expensive error, so the model that catches the most real failures leads.
2. **Tie-break - validation F1.** Where models fall within 1 percentage point of the best recall, the difference is inside the noise band of a 222-failure validation set (a single failure moves recall by 0.45 points). In that case we prefer the model with the better precision/recall balance, since it achieves comparable protection with fewer wasted inspections.
3. **Sanity check - the generalisation gap and total cost.** The selected model must not show a wide train-validation gap, and must sit at the low-cost end of the table.

In [51]:
# Applying the decision rule
best_recall = comparison_ranked.loc[0, "Val_Recall"]
TOLERANCE = 0.01

contenders = comparison_ranked[comparison_ranked["Val_Recall"] >= best_recall - TOLERANCE].copy()

print(f"Highest validation recall achieved : {best_recall:.4f}")
print(f"Models within {TOLERANCE:.0%} of that recall (statistical ties):")
display(contenders[["Model", "Configuration", "Val_Recall", "Val_Precision",
                    "Val_F1", "Overfit_Gap"]])

# Among the statistical ties, prefer the best precision/recall balance
best_row = contenders.sort_values("Val_F1", ascending=False).iloc[0]
best_model_name = best_row["Model"]
final_model = trained_models[best_model_name]

print("\n" + "=" * 80)
print(f"FINAL MODEL SELECTED : {best_model_name}")
print("=" * 80)
print(f"Configuration        : {best_row['Configuration']}")
print(f"Validation recall    : {best_row['Val_Recall']:.4f}")
print(f"Validation precision : {best_row['Val_Precision']:.4f}")
print(f"Validation F1        : {best_row['Val_F1']:.4f}")
print(f"Validation accuracy  : {best_row['Val_Accuracy']:.4f}")
print(f"Train-val recall gap : {best_row['Overfit_Gap']:.4f}")

final_cost = cost_df[cost_df["Model"] == best_model_name].iloc[0]
print(f"\nValidation cost      : {final_cost['Total_Cost']:,} units "
      f"({final_cost['Saving_Percent']}% saving vs no model)")

**Reasoning for the final model choice**

The decision rule selected **Model 7** (3 hidden layers of 128, 64 and 32 neurons, Adam, dropout 0.3, class weights), with validation recall **0.8964**, precision **0.9128**, F1 **0.9045** and a validation cost of **8,365 units - a 62.3% saving against the no-model benchmark**.

The selection was genuinely close, and it is worth being explicit about how it was resolved:

- **Two models tied exactly on the primary metric.** Models 6 and 7 both achieved validation recall of 0.8964, catching the identical 199 of 222 failures. Under our pre-committed rule, ties within one percentage point of the best recall are broken on **validation F1**, where Model 7 leads narrowly (0.9045 against 0.9025).
- **The margin is immaterial in business terms.** The two models differ by a single false positive and **5 cost units out of 22,200**. Model 6 would have been an equally defensible choice, and had we instead broken the tie on the generalisation gap or on architectural parsimony, **Model 6 would have won** (gap 0.0372 against 0.0450, with roughly a third of the parameters). We record this openly rather than presenting a marginal result as decisive.
- **What is decisive is the configuration, not the depth.** Every model in the top tier combines **Adam with class weights and dropout**. Model 4 demonstrated that Adam plus class weights *without* regularisation overfits badly (gap 0.1148); Models 6 and 7 show that adding dropout repairs it. **That combination - not the layer count - is the real finding.**
- **The choice is confirmed economically.** Model 7 sits at the lowest-cost end of the validation cost table, and the cost ranking across all eight models moves in lockstep with the recall ranking. This is independent, money-denominated confirmation that **recall was the correct metric to optimise.**
- **The generalisation gap is acceptable at 0.0450**, so the measured recall is a credible estimate of production performance rather than an artefact of memorising the 888 training failures. The held-out test results in the next section bear this out.

**A note on Model 5.** Model 5 achieved the best F1 (0.9267) and the best accuracy (0.9922) of any model built, on the strength of exceptional precision (0.9751). It is not our choice because its recall of 0.8829 trails the leaders, and **ReneWind's cost structure selects on recall**: the 3 extra failures Model 7 catches are worth far more than the 14 extra inspections it triggers. Under a different cost structure - one where inspection capacity was the binding constraint - Model 5 would be the right answer.

### Performance of the final model on the test set

Now, let's check the performance of the final model on the test set.

This is the first and only time `Test.csv` influences any decision in this notebook. Its 5,000 observations have been imputed and scaled using the parameters learned from the training split alone, and no model was chosen, tuned or stopped based on them. **The score below is therefore an honest estimate of production performance.**

In [52]:
print("=" * 80)
print(f"FINAL MODEL ({best_model_name}) EVALUATED ON THE HELD-OUT TEST SET")
print("=" * 80)

test_scores = get_metrics(final_model, X_test_final, y_test_final)
val_scores_final = get_metrics(final_model, X_val_final, y_val_final)

final_summary = pd.DataFrame({
    "Validation": [val_scores_final[m] for m in ["Accuracy", "Recall", "Precision", "F1"]],
    "Test": [test_scores[m] for m in ["Accuracy", "Recall", "Precision", "F1"]],
}, index=["Accuracy", "Recall", "Precision", "F1"]).round(4)
final_summary["Difference"] = (final_summary["Test"] - final_summary["Validation"]).round(4)

display(final_summary)

print(f"\nTest set contains {int(y_test_final.sum())} actual failures "
      f"out of {len(y_test_final)} generators ({y_test_final.mean()*100:.2f}%).")

In [53]:
plot_confusion(final_model, X_test_final, y_test_final,
               f"{best_model_name} - TEST SET (final, held-out)")

In [54]:
# The business consequence of the final model on the test set
test_cost = cost_breakdown(final_model, X_test_final, y_test_final)
test_no_model = int(y_test_final.sum()) * COST_REPLACEMENT

print("=" * 78)
print("BUSINESS IMPACT ON THE 5,000 HELD-OUT TEST GENERATORS")
print("=" * 78)
print(f"\nWITHOUT a predictive model (current state):")
print(f"  {int(y_test_final.sum())} failures all run to destruction")
print(f"  Cost = {int(y_test_final.sum())} x {COST_REPLACEMENT} = {test_no_model:,} units")

print(f"\nWITH the final model ({best_model_name}):")
print(f"  {test_cost['TP']:>4} failures caught early     -> repair      "
      f"= {test_cost['TP'] * COST_REPAIR:,} units")
print(f"  {test_cost['FN']:>4} failures missed           -> replacement "
      f"= {test_cost['FN'] * COST_REPLACEMENT:,} units")
print(f"  {test_cost['FP']:>4} healthy units inspected   -> inspection  "
      f"= {test_cost['FP'] * COST_INSPECTION:,} units")
print(f"  {test_cost['TN']:>4} healthy units left alone  -> no cost")
print(f"  {'':>4} {'':<25}    TOTAL       = {test_cost['Total_Cost']:,} units")

saving = test_no_model - test_cost["Total_Cost"]
print(f"\nNET SAVING = {saving:,} units "
      f"({saving / test_no_model * 100:.1f}% reduction in failure-related cost)")
print(f"Failures successfully detected: {test_cost['TP']} of {int(y_test_final.sum())} "
      f"({test_cost['TP'] / int(y_test_final.sum()) * 100:.1f}%)")

### Tuning the decision threshold - a further optimisation for the business

Every model above converts its predicted probability into a decision using the default cut-off of 0.5. But 0.5 carries no special significance for ReneWind: it implicitly assumes that a false alarm and a missed failure are equally costly, which we know is false.

There are two competing forces acting on where the optimal cut-off should sit, and they pull in **opposite** directions:

- The **cost asymmetry** pushes the threshold *down*. A missed failure costs 100 units against only 5 for a false alarm, so trading inspections for replacements is profitable.
- The **class weighting already applied during training** pushes the effective operating point *up*. By weighting failures 17 times more heavily in the loss, we have already biased the model towards predicting failure, so its raw probabilities are systematically inflated relative to true failure likelihood.

Because these two effects oppose one another, **the cost-optimal threshold cannot be reasoned out in advance - it has to be measured.** We therefore sweep the threshold and locate the minimum empirically **on the validation set** (never on the test set, which would compromise its independence), then apply the chosen value to the test set.

In [55]:
# Sweeping the decision threshold on the VALIDATION set
thresholds = np.arange(0.05, 0.96, 0.05)
sweep = []
for t in thresholds:
    b = cost_breakdown(final_model, X_val_final, y_val_final, threshold=t)
    b["Threshold"] = round(t, 2)
    b["Recall"] = recall_score(y_val_final,
                               (final_model.predict(X_val_final, verbose=0).ravel() >= t).astype(int),
                               zero_division=0)
    sweep.append(b)

sweep_df = pd.DataFrame(sweep)[["Threshold", "TN", "FP", "FN", "TP", "Recall", "Total_Cost"]]
sweep_df["Recall"] = sweep_df["Recall"].round(4)

best_threshold = float(sweep_df.loc[sweep_df["Total_Cost"].idxmin(), "Threshold"])
cost_at_default = int(sweep_df.loc[sweep_df["Threshold"] == 0.5, "Total_Cost"].iloc[0])
cost_at_best = int(sweep_df["Total_Cost"].min())

display(sweep_df)

print(f"\nCost-minimising threshold on the validation set : {best_threshold}")
print(f"  Cost at the default 0.5 threshold : {cost_at_default:,} units")
print(f"  Cost at the tuned threshold       : {cost_at_best:,} units")
print(f"  Further saving from tuning alone   : {cost_at_default - cost_at_best:,} units")

In [56]:
# Visualising the cost-recall trade-off across thresholds
fig, ax1 = plt.subplots(figsize=(11, 5.5))

ax1.plot(sweep_df["Threshold"], sweep_df["Total_Cost"], "o-",
         color="#C44E52", linewidth=2, label="Total maintenance cost")
ax1.axvline(best_threshold, color="green", linestyle="--", linewidth=1.6,
            label=f"Cost-optimal threshold = {best_threshold}")
ax1.axvline(0.5, color="grey", linestyle=":", linewidth=1.6, label="Default threshold = 0.5")
ax1.set_xlabel("Decision threshold"); ax1.set_ylabel("Total cost (units)", color="#C44E52")
ax1.tick_params(axis="y", labelcolor="#C44E52")

ax2 = ax1.twinx()
ax2.plot(sweep_df["Threshold"], sweep_df["Recall"], "s--",
         color="#4C72B0", linewidth=2, label="Recall")
ax2.set_ylabel("Recall", color="#4C72B0")
ax2.tick_params(axis="y", labelcolor="#4C72B0")
ax2.grid(False)

ax1.legend(loc="upper center")
plt.title("Choosing the decision threshold: cost vs recall (validation set)", fontsize=13)
plt.tight_layout()
plt.show()

In [57]:
# Applying the validation-tuned threshold to the held-out test set
tuned = cost_breakdown(final_model, X_test_final, y_test_final, threshold=best_threshold)
default = cost_breakdown(final_model, X_test_final, y_test_final, threshold=0.5)

compare_thresh = pd.DataFrame({
    f"Default (0.5)": [default["TP"], default["FN"], default["FP"], default["Total_Cost"]],
    f"Tuned ({best_threshold})": [tuned["TP"], tuned["FN"], tuned["FP"], tuned["Total_Cost"]],
}, index=["Failures caught (TP)", "Failures missed (FN)",
          "False alarms (FP)", "Total cost (units)"])

print("EFFECT OF THRESHOLD TUNING ON THE HELD-OUT TEST SET")
display(compare_thresh)

r_default = recall_score(y_test_final,
    (final_model.predict(X_test_final, verbose=0).ravel() >= 0.5).astype(int), zero_division=0)
r_tuned = recall_score(y_test_final,
    (final_model.predict(X_test_final, verbose=0).ravel() >= best_threshold).astype(int), zero_division=0)

print(f"\nTest recall at the default threshold : {r_default:.4f}")
print(f"Test recall at the tuned threshold   : {r_tuned:.4f}")
print(f"Cost difference on the test set      : "
      f"{default['Total_Cost'] - tuned['Total_Cost']:,} units saved by tuning")

**Observations**

- **The cost curve is clearly U-shaped, exactly as the opposing forces described above imply.** At a threshold of 0.05 the model flags almost everything - 1,229 false positives - and total cost reaches 13,645 units. At 0.95 it becomes too conservative, missing 27 failures, and cost climbs to 8,570. The minimum sits between these extremes.
- **The cost-minimising threshold is 0.50 - the default.** Total cost at that point is 8,365 units, and the sweep reports a further saving from tuning of **0 units**. The same holds on the test set: identical recall (0.8794), identical cost, **0 units saved by tuning.**
- **This is a genuine and somewhat elegant result rather than a null one.** The two forces we identified are real and they are roughly equal in magnitude: the 20:1 cost asymmetry between a replacement and an inspection pushes the optimal cut-off *down*, while the 17:1 class weighting already applied during training has inflated the model's probabilities and pushes the effective operating point *up*. **The two effects very nearly cancel, and the optimum lands on the default.** Had we not applied class weights, the cost-optimal threshold would sit well below 0.5 - Model 5's unweighted probabilities would need exactly that adjustment.
- **The practically important finding is how flat the curve is.** Total cost stays within 100 units of the minimum across the entire range from 0.35 to 0.75 (8,365 to 8,465). **The model's economics are highly insensitive to where the threshold is set within that band**, which is excellent news operationally: ReneWind can move the dial to suit inspection capacity without materially damaging cost performance.
- Note that the cost-optimal threshold does **not** coincide with maximum recall. Recall peaks at 0.9459 at a threshold of 0.05, where cost is 63% *higher* than at the optimum. This is an important qualification to our metric choice: **recall is the right metric for selecting between models, but realised cost is the right criterion for setting the operating point of the chosen model.** The two questions are distinct and deserve different answers.
- Methodologically, the threshold was identified on the **validation set** and only then applied to the test set, so no test-set information influenced the decision.

# **Actionable Insights and Recommendations**

## Key insights from the analysis

**1. The final model detects 88% of generator failures and cuts failure-related maintenance cost by 61%.**
On the 5,000 held-out test generators, Model 7 caught **248 of 282 real failures** (recall 0.8794) at a precision of 0.9018, missing 34 and raising only 27 false alarms across 4,718 healthy units. In cost terms this converts a **28,200-unit** reactive bill into **10,975 units - a saving of 17,225 units, or 61.1%.** Validation performance (recall 0.8964) transferred to the test set with a drop of only 0.017, so this is a dependable estimate rather than an optimistic one.

**2. Failures are rare, and that rarity - not the modelling technique - is the central challenge.**
Only **5.55% of observations are failures** (1,110 of 20,000). This one fact shaped every decision: it disqualified accuracy as a metric, made stratified splitting essential, and set the ceiling on what any model here can achieve. A model predicting "no failure" for every generator would score 94.45% accuracy and be worth nothing.

**3. No single technique dominated - performance came from combining three of them, and the combination had to be balanced.**
This was the most instructive finding, and it contradicts the intuition that one should simply apply the most powerful lever available:

| Lever | Effect on validation recall |
|---|---|
| Adding a second hidden layer (Model 0 → 1) | +4.1 points |
| Switching SGD → Adam (Model 1 → 2) | +4.1 points |
| Adding dropout (Model 2 → 5) | +2.7 points |
| Adding class weights, **under SGD** (Model 1 → 3) | **+7.2 points** |
| Adding class weights, **under Adam** (Model 2 → 4) | **+1.8 points only** |

Class weighting was the most powerful single lever under SGD but delivered barely a quarter of that benefit under Adam - and **Model 4 (Adam + class weights, unregularised) overfitted severely**, with a train-validation gap of 0.1148, more than triple any other model. Adding dropout repaired it, cutting the gap to 0.0372 and lifting recall to the study maximum. **The lesson is that individually beneficial techniques do not simply add up; Adam's optimisation strength and class weighting's aggression both push in the same direction and require regularisation to remain stable.**

**4. More depth bought nothing - the constraint is data, not model capacity.**
Model 7 (three hidden layers, 15,617 parameters) and Model 6 (two hidden layers, 4,737) achieved **identical validation recall of 0.8964**, catching the same 199 failures and missing the same 23. More than tripling the parameter count produced no additional detections. With only **888 failure examples** in the training split, there is a hard ceiling on how much the minority-class structure can be learned. **Further model complexity has nothing left to fit.**

**5. Accuracy and business value point in opposite directions here.**
Model 5 recorded the **highest accuracy (0.9922) and highest F1 (0.9267)** of any model built, yet it is not the model we selected, because its recall trails the leaders. Meanwhile Model 3 improved on Model 1 in business terms while its accuracy *fell* from 0.9890 to 0.9840. Any dashboard reporting accuracy for this system will systematically reward the wrong behaviour.

**6. The economics overwhelmingly favour aggressive detection, and the cost table proves the metric choice.**
Catching a failure early converts a **100-unit replacement into a 30-unit repair - a 70-unit saving** - while a false alarm costs only **5 units**. **One correctly caught failure therefore pays for fourteen unnecessary inspections.** The validation cost ranking across all eight models moved in lockstep with the recall ranking, from Model 7 at 8,365 units down to the baseline at 10,185. That is independent, money-denominated confirmation that recall was the right metric to optimise.

**7. The default 0.5 decision threshold is already cost-optimal - and the cost curve is remarkably flat.**
We swept the threshold from 0.05 to 0.95 and found the minimum at **exactly 0.50**, with a further saving from tuning of **0 units**. This is a real result, not a null one: the 20:1 cost asymmetry pushes the optimal cut-off down, while the 17:1 class weighting already applied during training pushes it up, and **the two effects very nearly cancel.** More usefully, total cost stays within 100 units of the minimum across the whole range **0.35 to 0.75**, so the operating point can be moved substantially without materially hurting cost performance.

## Business recommendations

**1. Deploy the selected model as a screening layer, not an autonomous decision-maker.**
Use it to rank the fleet by failure probability each cycle and route flagged units to inspection. On test-set rates, ReneWind would inspect roughly **275 of every 5,000 generators** and find a genuine developing fault in about 90% of them. The model decides *where to look first*; the engineer decides *what to do*.

**2. Operate at the default 0.5 threshold, and treat the 0.35-0.75 band as a capacity dial.**
Our analysis confirms 0.5 is cost-optimal, so no adjustment is needed at launch. Because the cost curve is nearly flat across 0.35-0.75, the threshold can nonetheless be moved within that band to manage workload - **towards 0.35 during peak generation season**, when downtime is most expensive and catching more failures is worth the extra inspections, and **towards 0.75 when inspection crews are stretched.** Outside that band cost rises steeply, so it should be treated as a hard operating boundary. Re-run the sweep after each retraining, since the optimum depends on the class weighting in force.

**3. Replace accuracy with recall and realised cost in all reporting.**
Track (a) **recall** - the share of actual failures detected, (b) the **count of false negatives**, and (c) **realised maintenance cost** against the no-model benchmark. Explicitly exclude accuracy from executive dashboards for this system: our own results show the highest-accuracy model is not the most valuable one.

**4. Prioritise collecting more labelled failure events over building bigger models.**
This is our strongest technical recommendation, and Model 7 is the evidence: ten times the baseline capacity produced zero additional detections. The ceiling is set by having only 888 training failures. **Every additional confirmed failure event is worth more than any architectural change.** Establish a disciplined process for logging every failure and every confirmed-healthy inspection outcome.

**5. Instrument the false positives to turn them into an asset.**
Each of the 27 inspections that found nothing costs 5 units but generates a valuable labelled record - an example the current model got wrong, which is the most informative kind of training data available. Capture sensor readings and inspection outcome for every flagged unit.

**6. Establish quarterly retraining with drift monitoring.**
Components age, sensors drift, and new units enter the fleet, so the mapping from sensor readings to failure will shift. Retrain quarterly and monitor incoming sensor distributions against the training distribution, so a material shift triggers investigation rather than silent degradation.

**7. Recover the ciphered feature mapping internally.**
Our EDA identified the variables carrying the strongest failure signal - **`V18` (Cohen's d = -1.34), `V21`, `V15`, `V7`, `V16` and `V39`**, all with large effect sizes above 1.0. ReneWind holds the key to what these represent. **Mapping them back to physical components would convert a black-box predictor into a diagnostic tool** - not just *"this generator will fail"* but *"this generator will fail, and the signal is coming from the gearbox."* Crews could then arrive with the right parts, shortening repair time and cutting the repair cost itself.

**8. Quantify the true cost ratios and re-optimise.**
Our analysis used illustrative relative costs (100 / 30 / 5) because only the ordering was supplied. ReneWind's finance team holds the real figures. Substituting them would re-rank the models on true economics and re-derive the optimal threshold - **a small analytical task with a direct, measurable financial return.** Given how flat we found the cost curve to be, we would expect the operating point to prove robust, but this should be verified rather than assumed.

## Expected benefit

On the 5,000 held-out test generators, the final model reduced failure-related maintenance cost from **28,200 units to 10,975 units - a 61.1% reduction** - relative to ReneWind's current reactive position where every failure runs to destruction. It achieved this by converting **248 catastrophic, unplanned replacements into scheduled, inexpensive repairs**, at the price of 27 inspections that found nothing.

Scaled across the operating fleet, this is precisely the operational efficiency the U.S. Department of Energy guidance identifies as the core value of predictive maintenance. The residual **34 missed failures per 5,000 generators** define the remaining opportunity - and our analysis indicates that closing that gap depends on **accumulating more labelled failure data**, not on further model engineering.